# B-RANKO Quickstart

This notebook walks through the two main public workflows for the released B-RANKO model bundle:

1. de novo RNA generation
2. representation extraction with covariance pooling

The example uses `branko_mega_cpool128.ckpt`, which includes the covariance-pooling head required for sequence-level embeddings.

## 1. Setup

### 1.1 Imports and model path

In [1]:
from pathlib import Path

import torch

from branko import BidirectionalRNAModel

MODEL_PATH = Path('../weights/branko_mega_cpool128.ckpt')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### 1.2 Load the pretrained model

In [2]:
model = BidirectionalRNAModel.from_checkpoint(MODEL_PATH, device=DEVICE)
model.eval()

print(f'Device: {DEVICE}')
print(f'Covariance pooler available: {model.covariance_pooler is not None}')

Device: cuda
Covariance pooler available: True


## 2. De Novo Generation

### 2.1 Sample fixed-length RNA sequences

In [3]:
generated_sequences = [
    model.generate(sequence_length=40, strategy='random', temperature=1.0)
    for _ in range(5)
]
generated_sequences

['UGCUGAGGAGUUCCACAAUAAACUCAACAGUUCUGUAGCA',
 'ACGCGUCAGCGACCCCCGUCGGUCAGGCGGGACUACCCGC',
 'UCGUGACCCCAGGUCAGGCGGGAUACCCGCUGAACUUAAG',
 'GUGUUCGCGCAACGAAUACCACAAGCACUCAAUUUUGAAU',
 'GCGACCCUGGGCAGGGUGACCUGAUCUCGGAAGCUAAGCA']

## 3. Representation Extraction

### 3.1 Define example RNA sequences

In [4]:
sequences = [
    'AUGCAUGCAUGC',
    'GGGAAAUUUCCC',
    'CCGAUUGGCAUAAUGC',
]
sequences

['AUGCAUGCAUGC', 'GGGAAAUUUCCC', 'CCGAUUGGCAUAAUGC']

### 3.2 Extract covariance-pooled sequence embeddings

In [8]:
encoded = model.encode(sequences, pooling='covariance')

print('Last hidden state shape:', tuple(encoded.last_hidden_state.shape))
print('Covariance-pooled embedding shape:', tuple(encoded.pooler_output.shape))

Last hidden state shape: (3, 18, 640)
Covariance-pooled embedding shape: (3, 640)
